# IAM Roles Anywhere — lab (automation)

This notebook automates a standard IAM Roles Anywhere setup.
It performs the steps programmatically (easy-rsa CLI + `boto3`) so the whole thing is
repeatable and easy to clean up.

## What the notebook does
1. Builds its own CA using **easy-rsa** (external CA — as in the lab).
2. Issues a cert/key for a single user: **sympuser**.
3. Creates **one role** (`symp_role`) in AWS with a trust policy restricted by the condition
   `x509Subject/CN = sympuser` (only this certificate can assume the role).
4. Attaches the managed policy **`arn:aws:iam::aws:policy/job-function/ViewOnlyAccess`** to the role.
5. Registers the CA as a **trust anchor** and creates an IAM Roles Anywhere **profile**.
6. Downloads `aws_signing_helper` and **generates temporary credentials**, verifying
   that `sympuser` assumes `symp_role` (`sts:GetCallerIdentity`).

## Requirements
- `python3` + `boto3` (available in the environment), `git`, `jq`, `curl`.
- Properly configured AWS access (`aws configure` / env vars / profile) with permissions for
  IAM (roles/policies) and IAM Roles Anywhere (trust anchors/profiles).
- Runtime environment: **macOS/Linux**. The notebook detects OS/CPU and picks the matching helper binary.

> **Cost/security note:** the notebook creates real IAM resources in your AWS account.
> The CA private key and the user key are created locally under `pki/`. Do not commit them.
> At the end there is a **Cleanup** section that removes the created AWS resources.

## 0. Parameters and AWS identity check

Set the region and names. `AccountID` is detected automatically via STS — this confirms
that AWS access works before we create anything.

In [ ]:
import os
import json
import subprocess
import platform
from pathlib import Path

import boto3
from botocore.exceptions import ClientError

# ---- Lab parameters (adjust as needed) ----
REGION = "eu-central-1"

USER = "sympuser"                 # the only lab user (certificate CN)

ROLE = "symp_role"                 # the only role — trust policy for the trust anchor

TRUST_ANCHOR_NAME = "sympozium-anchor"
PROFILE_NAME = "sympozium-profile"

# Managed policy granting permissions to the role (task requirement)
PERMISSION_POLICY_ARN = "arn:aws:iam::aws:policy/job-function/ViewOnlyAccess"

# Notebook working directory (relative to where you start the kernel)
WORKDIR = Path("work").resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)
EASYRSA_DIR = WORKDIR / "easy-rsa" / "easyrsa3"
PKI_DIR = EASYRSA_DIR / "pki"

# ---- AWS session + AccountID detection ----
session = boto3.Session(region_name=REGION)
sts = session.client("sts")
ident = sts.get_caller_identity()
ACCOUNT_ID = ident["Account"]

print("AWS OK")
print("  Account:", ACCOUNT_ID)
print("  Caller :", ident["Arn"])
print("  Region :", REGION)
print("  Workdir:", WORKDIR)

### Helper for running shell commands

A few steps (easy-rsa, `aws_signing_helper`) are CLI tools. This helper runs them
and shows the output, aborting on error.

In [ ]:
def run(cmd, cwd=None, env=None, check=True, capture=False):
    """Runs a command (list of args). Returns a CompletedProcess."""
    print(f"$ {' '.join(cmd)}" + (f"   (cwd={cwd})" if cwd else ""))
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command exited with code {result.returncode}: {' '.join(cmd)}")
    return result

## 1–2. Create and initialize the CA (easy-rsa)

Clone easy-rsa, `init-pki`, `build-ca nopass`.
We run easy-rsa non-interactively (`--batch`) so it does not prompt for a Common Name.

In [ ]:
# Step 1: clone easy-rsa (idempotently)
easyrsa_repo = WORKDIR / "easy-rsa"
if not easyrsa_repo.exists():
    run(["git", "clone", "https://github.com/OpenVPN/easy-rsa.git", str(easyrsa_repo)])
else:
    print("easy-rsa already cloned:", easyrsa_repo)

EASYRSA = "./easyrsa"

# Step 2: init-pki + build-ca (nopass, batch = no interactive prompts)
if not PKI_DIR.exists():
    run([EASYRSA, "--batch", "init-pki"], cwd=EASYRSA_DIR)
    run([EASYRSA, "--batch", "--req-cn=sympozium-roles-anywhere-ca", "build-ca", "nopass"], cwd=EASYRSA_DIR)
else:
    print("PKI already initialized:", PKI_DIR)

print("\nCA cert:", PKI_DIR / "ca.crt")

## 3–4. Issue a cert/key for the user

Steps **3–4**: `build-client-full sympuser nopass`.
The certificate Common Name (CN) = the user name (`sympuser`).

In [ ]:
def build_client(name):
    cert = PKI_DIR / "issued" / f"{name}.crt"
    if cert.exists():
        print(f"Cert for {name} already exists: {cert}")
        return
    run([EASYRSA, "--batch", "build-client-full", name, "nopass"], cwd=EASYRSA_DIR)

build_client(USER)

# Paths to cert/key — needed when generating credentials
CERTS = {
    USER: {
        "cert": PKI_DIR / "issued" / f"{USER}.crt",
        "key": PKI_DIR / "private" / f"{USER}.key",
    },
}
for u, paths in CERTS.items():
    print(u, "->", paths["cert"].exists(), paths["key"].exists())

## 5. Collect the CA certificate

Step **5**: read `ca.crt` — we register this PEM content in the trust anchor (steps 10–12).

In [ ]:
CA_CERT_PATH = PKI_DIR / "ca.crt"
CA_CERT_PEM = CA_CERT_PATH.read_text()
print(CA_CERT_PEM)

## 6–7. Trust policy and role (`symp_role`)

Steps **6–7**: a trust policy allowing `rolesanywhere.amazonaws.com` to call
`sts:AssumeRole`, `sts:TagSession`, `sts:SetSourceIdentity`, restricted by the condition
`StringEquals: aws:PrincipalTag/x509Subject/CN = sympuser` — only a certificate
with CN=sympuser can assume the role. We create the role and attach `ViewOnlyAccess`.

In [ ]:
iam = session.client("iam")

TRUST_POLICY = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": ["rolesanywhere.amazonaws.com"]},
            "Action": [
                "sts:AssumeRole",
                "sts:TagSession",
                "sts:SetSourceIdentity",
            ],
            "Condition": {
                "StringEquals": {
                    "aws:PrincipalTag/x509Subject/CN": USER
                }
            },
        }
    ],
}


def ensure_role(role_name, trust_policy):
    """Creates the role (idempotently) and attaches ViewOnlyAccess. Returns the ARN."""
    try:
        resp = iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description="Sympozium IAM Roles Anywhere",
        )
        arn = resp["Role"]["Arn"]
        print("Created role:", arn)
    except ClientError as e:
        if e.response["Error"]["Code"] == "EntityAlreadyExists":
            arn = iam.get_role(RoleName=role_name)["Role"]["Arn"]
            # update the trust policy in case it changed
            iam.update_assume_role_policy(
                RoleName=role_name, PolicyDocument=json.dumps(trust_policy)
            )
            print("Role exists, trust policy updated:", arn)
        else:
            raise
    iam.attach_role_policy(RoleName=role_name, PolicyArn=PERMISSION_POLICY_ARN)
    print("  Attached policy:", PERMISSION_POLICY_ARN)
    return arn


ROLE_ARN = ensure_role(ROLE, TRUST_POLICY)

## 10–12. Create the trust anchor (register the CA)

Steps **10–12**: we register the external CA (`ca.crt`) as a trust anchor of type
`CERTIFICATE_BUNDLE`. We save the `AnchorID`/ARN (equivalent to “Record the anchor ID”).

In [ ]:
ra = session.client("rolesanywhere", region_name=REGION)


def find_trust_anchor(name):
    paginator = ra.get_paginator("list_trust_anchors")
    for page in paginator.paginate():
        for ta in page["trustAnchors"]:
            if ta["name"] == name:
                return ta
    return None


existing = find_trust_anchor(TRUST_ANCHOR_NAME)
if existing:
    TRUST_ANCHOR = existing
    print("Trust anchor already exists:", existing["trustAnchorId"])
else:
    resp = ra.create_trust_anchor(
        name=TRUST_ANCHOR_NAME,
        source={
            "sourceType": "CERTIFICATE_BUNDLE",
            "sourceData": {"x509CertificateData": CA_CERT_PEM},
        },
        enabled=True,
    )
    TRUST_ANCHOR = resp["trustAnchor"]
    print("Created trust anchor:", TRUST_ANCHOR["trustAnchorId"])

TRUST_ANCHOR_ARN = TRUST_ANCHOR["trustAnchorArn"]
TRUST_ANCHOR_ID = TRUST_ANCHOR["trustAnchorId"]
print("  ARN:", TRUST_ANCHOR_ARN)
print("  Status:", TRUST_ANCHOR.get("enabled"))

## 13–15. Create the profile and assign the role

Steps **13–15**: an IAM Roles Anywhere profile to which we assign the role
`symp_role` (from step 7). We save the `ProfileID`/ARN.

In [ ]:
def find_profile(name):
    paginator = ra.get_paginator("list_profiles")
    for page in paginator.paginate():
        for p in page["profiles"]:
            if p["name"] == name:
                return p
    return None


role_arns = [ROLE_ARN]
existing_profile = find_profile(PROFILE_NAME)
if existing_profile:
    PROFILE = existing_profile
    print("Profile already exists:", existing_profile["profileId"])
    ra.update_profile(profileId=PROFILE["profileId"], roleArns=role_arns)
    print("  Updated role list")
else:
    resp = ra.create_profile(
        name=PROFILE_NAME,
        roleArns=role_arns,
        enabled=True,
    )
    PROFILE = resp["profile"]
    print("Created profile:", PROFILE["profileId"])

PROFILE_ARN = PROFILE["profileArn"]
PROFILE_ID = PROFILE["profileId"]
print("  ARN:", PROFILE_ARN)
print("  Role:", role_arns)

## 16. Download `aws_signing_helper`

Step **16**: instead of cloning a helper script, we download AWS's official
`aws_signing_helper` binary (the Roles Anywhere credential helper) matched to OS/CPU.
It implements `credential-process` and returns temporary credentials.

In [ ]:
import hashlib

# Official AWS credential helper links (v1.8.5) + SHA256, per
# https://docs.aws.amazon.com/rolesanywhere/latest/userguide/credential-helper.html
# Key: (system, is_arm) -> (url, sha256)
HELPER_RELEASES = {
    ("Darwin", True): (
        "https://rolesanywhere.amazonaws.com/releases/1.8.5/Aarch64/MacOS/Sonoma/aws_signing_helper",
        "ac4b656cd83ffde5a6e9e8f2317ffb90e036c9bb704cc80faa6aee414b55915a",
    ),
    ("Darwin", False): (
        "https://rolesanywhere.amazonaws.com/releases/1.8.5/X86_64/MacOS/Sonoma/aws_signing_helper",
        "aab355e1e7468056be88a56bbfb030ea33ff32bef2ce20f5dd6a0b1cae5aae5a",
    ),
    ("Linux", True): (
        "https://rolesanywhere.amazonaws.com/releases/1.8.5/Aarch64/Linux/Amzn2023/aws_signing_helper",
        "3d131aa888cd56da446f9c6bb460b1f0569f6c7edc74eae6193a2fe3928883ba",
    ),
    ("Linux", False): (
        "https://rolesanywhere.amazonaws.com/releases/1.8.5/X86_64/Linux/Amzn2023/aws_signing_helper",
        "beec9ed1c492d93db809890f16713e3556353294b823c2184ad4e891f1b2b54d",
    ),
}

sys_name = platform.system()           # 'Darwin' | 'Linux'
machine = platform.machine().lower()   # 'arm64' | 'x86_64' | 'aarch64'
is_arm = machine in ("arm64", "aarch64")

key = (sys_name, is_arm)
if key not in HELPER_RELEASES:
    raise RuntimeError(
        f"No link defined for {sys_name}/{machine}. "
        "Check the AWS documentation and extend HELPER_RELEASES."
    )
url, expected_sha = HELPER_RELEASES[key]
HELPER_PATH = WORKDIR / "aws_signing_helper"


def sha256_of(path):
    h = hashlib.sha256()
    h.update(Path(path).read_bytes())
    return h.hexdigest()


need_download = (not HELPER_PATH.exists()) or (sha256_of(HELPER_PATH) != expected_sha)
if need_download:
    print("Downloading helper:", url)
    run(["curl", "-fSL", "-o", str(HELPER_PATH), url])
    HELPER_PATH.chmod(0o755)
else:
    print("Helper already downloaded and matches SHA256:", HELPER_PATH)

actual = sha256_of(HELPER_PATH)
assert actual == expected_sha, f"Helper SHA256 mismatch!\n expected: {expected_sha}\n actual:   {actual}"
print("SHA256 OK:", actual)

run([str(HELPER_PATH), "version"], check=False)

## 17–19. Generate credentials for sympuser → `symp_role`

Steps **17–19**: the helper signs the request with the `sympuser` certificate and assumes
the `symp_role` role. This corresponds to the success in step 19 (“successfully assumed role”).
The function below returns the credentials and verifies them via `sts:GetCallerIdentity`.

In [ ]:
def get_credentials(user, role_arn):
    """Calls aws_signing_helper credential-process. Returns a dict with credentials,
    or prints the stderr (e.g. AccessDenied) and returns None in that case."""
    cert = CERTS[user]["cert"]
    key = CERTS[user]["key"]
    cmd = [
        str(HELPER_PATH), "credential-process",
        "--certificate", str(cert),
        "--private-key", str(key),
        "--trust-anchor-arn", TRUST_ANCHOR_ARN,
        "--profile-arn", PROFILE_ARN,
        "--role-arn", role_arn,
    ]
    print(f"[{user}] -> {role_arn}")
    res = subprocess.run(cmd, text=True, capture_output=True)
    if res.returncode != 0:
        print("  ERROR:", res.stderr.strip())
        return None
    creds = json.loads(res.stdout)
    return creds


def whoami(creds):
    """Confirms the identity for the generated credentials (sts get-caller-identity)."""
    s = boto3.Session(
        aws_access_key_id=creds["AccessKeyId"],
        aws_secret_access_key=creds["SecretAccessKey"],
        aws_session_token=creds["SessionToken"],
        region_name=REGION,
    )
    who = s.client("sts").get_caller_identity()
    print("  Assumed:", who["Arn"])
    return who


creds_symp = get_credentials(USER, ROLE_ARN)
assert creds_symp, "sympuser should assume symp_role"
whoami(creds_symp)

## Summary: collected identifiers

The values you “record” manually in the original lab (anchor ID, profile ID, ARNs).
We also save them to `work/lab-outputs.json` — useful when wiring this into MCP in Sympozium.

In [ ]:
outputs = {
    "Region": REGION,
    "AccountID": ACCOUNT_ID,
    "TrustAnchorId": TRUST_ANCHOR_ID,
    "TrustAnchorArn": TRUST_ANCHOR_ARN,
    "ProfileId": PROFILE_ID,
    "ProfileArn": PROFILE_ARN,
    "Roles": {
        ROLE: ROLE_ARN,
    },
    "PermissionPolicy": PERMISSION_POLICY_ARN,
    "Certs": {u: {"cert": str(p["cert"]), "key": str(p["key"])} for u, p in CERTS.items()},
}
out_path = WORKDIR / "lab-outputs.json"
out_path.write_text(json.dumps(outputs, indent=2))
print(json.dumps(outputs, indent=2))
print("\nSaved:", out_path)

## Cleanup (optional) — remove the created AWS resources

Run this when the lab is no longer needed. It removes the profile, trust anchor, and role
(detaching the managed policy first). It does **not** delete the local PKI — do that manually
(`rm -rf work/`) when you decide to.

> Set `DO_CLEANUP = True` to execute.

In [ ]:
DO_CLEANUP = False

if DO_CLEANUP:
    # 1) profile
    p = find_profile(PROFILE_NAME)
    if p:
        ra.delete_profile(profileId=p["profileId"])
        print("Deleted profile:", p["profileId"])
    # 2) trust anchor
    ta = find_trust_anchor(TRUST_ANCHOR_NAME)
    if ta:
        ra.delete_trust_anchor(trustAnchorId=ta["trustAnchorId"])
        print("Deleted trust anchor:", ta["trustAnchorId"])
    # 3) role (detach policy, then delete)
    for r in (ROLE,):
        try:
            iam.detach_role_policy(RoleName=r, PolicyArn=PERMISSION_POLICY_ARN)
        except ClientError as e:
            print("  detach:", e.response["Error"]["Code"])
        try:
            iam.delete_role(RoleName=r)
            print("Deleted role:", r)
        except ClientError as e:
            print("  delete_role:", e.response["Error"]["Code"])
else:
    print("Cleanup skipped (DO_CLEANUP=False).")